# 5 · First linear solve

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve is pinned to a tested release (6.2.2606) for a reproducible build.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "ngsolve==6.2.2606", "anywidget"], check=True)

Preparation for drawing sparsity patterns:

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

mesh = Mesh(unit_square.GenerateMesh(maxh=0.4))

In [ ]:
### drawing facilities for sparsity patterns of matrices ...
def to_dense(ngmat, n):
    """An NGSolve sparse matrix as a dense n×n NumPy array (these are small, for drawing)."""
    rows, cols, vals = ngmat.COO()
    A = np.zeros((n, n))
    A[np.asarray(rows, int), np.asarray(cols, int)] = np.asarray(vals, float)
    return A

def spy_blocks(ax, A, cut, labels, colors, title):
    """`spy` A and shade the 2×2 block structure induced by a dof split at index `cut`."""
    n = A.shape[0]
    ax.spy(A, markersize=2.2, color="#334155")
    seg = [0, cut, n]
    for i in range(2):
        for j in range(2):
            x0, y0 = seg[j], seg[i]
            w, h = seg[j + 1] - seg[j], seg[i + 1] - seg[i]
            ax.add_patch(patches.Rectangle((x0 - .5, y0 - .5), w, h, facecolor=colors[i][j],
                                           alpha=.22, edgecolor=colors[i][j], lw=1.2))
            ax.text(x0 + w / 2 - .5, y0 + h / 2 - .5, labels[i][j], ha="center", va="center",
                    color=colors[i][j], fontsize=22, weight="bold")
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

## 1. The simplest variational problem — an L²-projection

To **project** a function $g$ onto a finite element space we solve $M\,\mathbf u=\mathbf b$ with the **mass matrix** $M_{ij}=\int\varphi_i\varphi_j$ and $b_i=\int g\,\varphi_i$. **Every** dof is an unknown — nothing is prescribed.

In [ ]:
g = sin(5 * x) * y
u,v = H1(mesh, order=3).TnT()
gfp = Solve(u*v*dx==g*v*dx)
print(f"L2 error ‖u−g‖ = {sqrt(Integrate((gfp - g)**2, mesh)):.1e}")

> **`u` and `v` (trial and test functions) are also `CoefficientFunctions`**
>
> They are proxies for the shape functions and allow to form `Integral`s and `SumOfIntegrals` (together with the `DifferentialSymbol`s `dx` and `ds`.

Linear variational problem decomposed into `BilinearForm` and `LinearForm`:

In [ ]:
fesP = H1(mesh, order=3)
u, v = fesP.TnT()
M = BilinearForm(u * v * dx).Assemble() # short for M = BilinearForm(fesP); M+=u*v*dx; M.Assemble()
f = LinearForm(g * v * dx).Assemble()   # short for f = LinearForm(fesP); f+=g*v*dx; f.Assemble()
gfp = GridFunction(fesP)
gfp.vec.data = M.mat.Inverse(fesP.FreeDofs()) * f.vec
print(f"projected {fesP.ndof} dofs;   L2 error ‖u−g‖ = {sqrt(Integrate((gfp - g)**2, mesh)):.1e}")

The matrix is **sparse** and **symmetric**: each basis function overlaps only its
neighbours, so $M$ has just a handful of entries per row.

In [ ]:
Mdense = to_dense(M.mat, fesP.ndof)
fig, ax = plt.subplots(figsize=(4.3, 4.3))
ax.spy(Mdense, markersize=2.2, color="#334155")
plt.show()

NGSolve has (sparse) direct solver:
* `sparsecholesky`: comes with `NGSolve` - suitable for s.p.d. systems
* `umfpack`: (often) comes with `NGSolve` - sparse systems (also non-symmetric, indefinite)
* `pardiso`: comes with mkl - sparse systems (faster than umfpack)
  
You specify it with `....Inverse(.., inverse="sparsecholesky")`.

## 2. Free dofs & Dirichlet dofs — a block system

Now solve **Poisson** $-\Delta u=f$ with $u=g$ on the boundary — the **variational equation**
$a(u,v)=f(v)$. 

* its left side is a **bilinear form** $a$
* its right side a **linear form** $f$
* `dirichlet` flag marks the boundary dofs **essential** — *prescribed*, not solved for — and `fes.FreeDofs()` is a `BitArray`, `True` on
the **free** dofs.

In [ ]:
fes = H1(mesh, order=3, dirichlet="bottom|left")
print("dirichlet region:", fes.GetDirichletRegion().Mask())
print("freedofs:\n", fes.FreeDofs())
fes = H1(mesh, order=3, dirichlet="bottom|right|top|left")
print("dirichlet region:", fes.GetDirichletRegion().Mask())
print("freedofs:\n", fes.FreeDofs())

We **assemble the full system** $A,\ \mathbf b$ — *every* dof, free or Dirichlet — and never
build a smaller matrix: 
* NGSolve sets matrices and vectors up *"with respect to all unknowns,
so they can be restricted to any group of unknowns later"* (i-tutorial 1.3).
* Split the dofs into **free** ($f$) and **Dirichlet** ($d$);
* want to solve for **free block** with the prescribed values $\mathbf u_d$ carried to the right-hand side:
$$ A_{ff}\,\mathbf u_f \;=\; \mathbf b_f - A_{fd}\,\mathbf u_d . $$

In the matrix *as assembled* the two kinds of dof are **interspersed**; reordering them
*free-first* (for illustration only — NGSolve never renumbers) makes the blocks line up:
$$A=\bigl(\begin{smallmatrix}A_{ff}&A_{fd}\\A_{df}&A_{dd}\end{smallmatrix}\bigr)$$

In [ ]:
u, v = fes.TnT()
a = BilinearForm(grad(u) * grad(v) * dx).Assemble()
f = LinearForm(1 * v * dx).Assemble()

free = fes.FreeDofs()
nf = free.NumSet()
print(f"{fes.ndof} dofs  =  {nf} free  +  {fes.ndof - nf} Dirichlet")

In [ ]:
# visualize sparsity pattern(s)
fmask = np.array([bool(free[i]) for i in range(fes.ndof)])
nf = int(fmask.sum())
perm = np.concatenate([np.where(fmask)[0], np.where(~fmask)[0]])     # free dofs first

GREEN, ORANGE, RED = "#10b981", "#f59e0b", "#ef4444"
A = to_dense(a.mat, fes.ndof)
rows, cols = np.nonzero(A)
entry_color = [GREEN if (fmask[i] and fmask[j]) else (RED if not (fmask[i] or fmask[j]) else ORANGE)
               for i, j in zip(rows, cols)]

fig, ax = plt.subplots(1, 2, figsize=(9.4, 4.8))
# (left) the matrix as assembled — entries coloured by block, dofs interspersed
ax[0].scatter(cols, rows, s=7, marker="s", c=entry_color)
ax[0].set_xlim(-0.5, fes.ndof - .5); ax[0].set_ylim(fes.ndof - .5, -0.5); ax[0].set_aspect("equal")
ax[0].set_xticks([]); ax[0].set_yticks([])
ax[0].set_title("(free·free green · dir·dir red · mixed orange)", fontsize=9)
# (right) reordered free-first → the blocks line up
spy_blocks(ax[1], A[perm][:, perm], nf,
           [["$A_{ff}$", "$A_{fd}$"], ["$A_{df}$", "$A_{dd}$"]], [[GREEN, ORANGE], [ORANGE, RED]],
           f"reordered: {nf} free + {fes.ndof - nf} Dirichlet")
plt.tight_layout(); plt.show()

The point is to do all this **without ever extracting $A_{ff}$**:
* **set** the boundary values into the solution ($\mathbf u_d$, via `Set(g, fes.GetDirichletRegion())` (or `Set(g, BND)`) );
* form the **residual** $\mathbf r=\mathbf b-A\,\mathbf u$ on the *whole* vector — in the free
  rows this is exactly $\mathbf b_f-A_{fd}\mathbf u_d$, the right-hand side above;
* apply **`A.Inverse(freedofs)`**, which inverts **only the free block** and leaves the
  Dirichlet rows and columns untouched, updating just $\mathbf u_f$:

So `A.Inverse(freedofs)` is the block solve $A_{ff}^{-1}$ done on the big matrix without
renumbering: the Dirichlet rows and columns are simply struck out (left), which is exactly
inverting the reordered free block $A_{ff}$ (right):

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.4, 4.9))
# (left) original numbering: assume every coupling (a full green matrix), then strike out the
# Dirichlet rows AND columns — exactly what Inverse(freedofs) skips, in place.
ax[0].add_patch(patches.Rectangle((-.5, -.5), fes.ndof, fes.ndof, facecolor=GREEN, alpha=.5))
for d in np.where(~fmask)[0]:
    ax[0].add_patch(patches.Rectangle((-.5, d - .5), fes.ndof, 1, facecolor="#94a3b8", alpha=.92))
    ax[0].add_patch(patches.Rectangle((d - .5, -.5), 1, fes.ndof, facecolor="#94a3b8", alpha=.92))
ax[0].set_xlim(-.5, fes.ndof - .5); ax[0].set_ylim(fes.ndof - .5, -.5); ax[0].set_aspect("equal")
ax[0].set_xticks([]); ax[0].set_yticks([])
ax[0].set_title("original numbering — strike out the\nDirichlet rows & columns (no renumbering)", fontsize=9)
# (right) reordered free-first: invert only the free block A_ff; the Dirichlet block is left alone.
ax[1].add_patch(patches.Rectangle((-.5, -.5), nf, nf, facecolor=GREEN, alpha=.5, edgecolor=GREEN, lw=1.6))
ax[1].text(nf / 2 - .5, nf / 2 - .5, "$A_{ff}^{-1}$", ha="center", va="center", color="#065f46", fontsize=22)
ax[1].add_patch(patches.Rectangle((nf - .5, -.5), fes.ndof - nf, fes.ndof, facecolor="#94a3b8", alpha=.55))
ax[1].add_patch(patches.Rectangle((-.5, nf - .5), nf, fes.ndof - nf, facecolor="#94a3b8", alpha=.55))
ax[1].set_xlim(-.5, fes.ndof - .5); ax[1].set_ylim(fes.ndof - .5, -.5); ax[1].set_aspect("equal")
ax[1].set_xticks([]); ax[1].set_yticks([])
ax[1].set_title("reordered — invert only the free block\n$A_{ff}$; the Dirichlet block stays untouched", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
g = sin(4 * x) * y
gfu = GridFunction(fes)
gfu.Set(g, definedon=fes.GetDirichletRegion())       # u_d : prescribe u = g on the boundary
r = f.vec - a.mat * gfu.vec                          # residual = b − A u  (carries A_fd u_d)
gfu.vec.data += a.mat.Inverse(free, inverse="sparsecholesky") * r   # solve on the free dofs - leave dirichlet dofs as is
Draw(gfu, mesh, "u")

**Reminder: We keep the full system!** 
* Restricting *later* is cheap and flexible
* Needs to be considered when setting up factorizations and preconditioner applications

> **Static condensation — the natural next step**
>
> A high-order space has many **internal** dofs that couple only to their own element; they can be
> **condensed out** locally (`condense=True`), shrinking the global solve to the *coupling* dofs and
> reconstructing the internals element-by-element afterwards — same answer, smaller system. The
> [**i-tutorial 1.4 — Static condensation**](https://docu.ngsolve.org/latest/i-tutorials/unit-1.4-staticcond/staticcond.html)
> walks through it in full.

**Next:** the solve above used a direct factorisation. Unit 6 — the close of Part I —
opens the **solver toolbox**: the iterative methods and preconditioners that scale to
problems a direct solver can no longer swallow.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("04-fespaces", "4 · A zoo of finite element spaces")
    _next = ("06-linear-solvers", "6 · The solver toolbox 🛠")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))